## STEP 1 Semantic Chunking(유사도 기반 청킹)

In [32]:
import re
from openai import OpenAI
import numpy as np
import pandas as pd
from dotenv import load_dotenv

load_dotenv()
client = OpenAI()
df = pd.read_excel("Articles_20260305_125404.xlsx")

text = df.iloc[0]['content']

# 문장 단위로 분리
# 한국어 문장은 보통 "~했다. ", "~이다. " 처럼 "다" + 공백으로 끝나니까, 다를 포함
# 정확도를 높이려면 kss 와 같은 한국어 문장 분리 라이브러리 사용
sentences = [s.strip() for s in re.split(r'[.!?\n다]\s', text) if len(s.strip()) > 10]
print(f"문장 수: {len(sentences)}")
for i, s in enumerate(sentences[:5]):
    print(f"  {i}: {s[:60]}")

문장 수: 18
  0: 당내 강경파, 대법원장 탄핵 공청회 개최
정청래 “사퇴도 타이밍… 거취 표명” 압박
  1: 집권여당 더불어민주당과 사법부 간 갈등이 악화일로를 걷고 있다
  2: 조희대 대법원장을 향한 민주당의 사퇴 압박이 거세지고 있다
  3: ‘사법개혁 3법’에 대해 반대의사를 표명한 조 대법원장을 놓고 민주당 정청래 대표가 “사법개혁 저항의 우두머
  4: 당내 강경파에서는 조 대법원장 탄핵까지 거론된다


In [33]:
# 전체 문장 임베딩
response = client.embeddings.create(
    input=sentences,
    model="text-embedding-3-small"
)
embeddings = [d.embedding for d in response.data]

# 인접 문장 간 코사인 유사도
def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

similarities = []
for i in range(len(embeddings) - 1):
    sim = cosine_similarity(embeddings[i], embeddings[i+1])
    similarities.append(sim)
    print(f"문장 {i} ↔ {i+1}: {sim:.4f}")

문장 0 ↔ 1: 0.3521
문장 1 ↔ 2: 0.5014
문장 2 ↔ 3: 0.5687
문장 3 ↔ 4: 0.4464
문장 4 ↔ 5: 0.2761
문장 5 ↔ 6: 0.4253
문장 6 ↔ 7: 0.5272
문장 7 ↔ 8: 0.4239
문장 8 ↔ 9: 0.3299
문장 9 ↔ 10: 0.5441
문장 10 ↔ 11: 0.5775
문장 11 ↔ 12: 0.4153
문장 12 ↔ 13: 0.3630
문장 13 ↔ 14: 0.5013
문장 14 ↔ 15: 0.3512
문장 15 ↔ 16: 0.2365
문장 16 ↔ 17: 0.2215


In [34]:
# 평균보다 낮은 지점에서 자르기
threshold = np.mean(similarities) - np.std(similarities)
print(f"임계값: {threshold:.4f}\n")

chunks = []
current_chunk = [sentences[0]]

for i, sim in enumerate(similarities):
    if sim < threshold:
        # 유사도 급락 → 여기서 자름
        chunks.append(" ".join(current_chunk))
        current_chunk = [sentences[i+1]]
        print(f"✂ 문장 {i}~{i+1} 사이에서 자름 (유사도: {sim:.4f})")
    else:
        current_chunk.append(sentences[i+1])

chunks.append(" ".join(current_chunk))  # 마지막 청크

print(f"\n총 청크 수: {len(chunks)}")
for i, chunk in enumerate(chunks):
    print(f"\n--- 청크 {i} ({len(chunk)}자) ---")
    print(chunk[:120])

임계값: 0.3064

✂ 문장 4~5 사이에서 자름 (유사도: 0.2761)
✂ 문장 15~16 사이에서 자름 (유사도: 0.2365)
✂ 문장 16~17 사이에서 자름 (유사도: 0.2215)

총 청크 수: 4

--- 청크 0 (227자) ---
당내 강경파, 대법원장 탄핵 공청회 개최
정청래 “사퇴도 타이밍… 거취 표명” 압박 집권여당 더불어민주당과 사법부 간 갈등이 악화일로를 걷고 있다 조희대 대법원장을 향한 민주당의 사퇴 압박이 거세지고 있다 ‘사법개혁

--- 청크 1 (753자) ---
연합뉴스
정 대표는 4일 국회에서 열린 최고위원회의 모두 발언에서 “조희대 사법부에서 수사를 방해하고 내란 수괴 윤석열과 잔당들에 대한 침대축구 재판을 통해 사법불신을 눈덩이처럼 키워온 것에 일말의 양심의 가책도 없

--- 청크 2 (13자) ---
정치적 부담이 상당한 데

--- 청크 3 (62자) ---
헌법재판소의 판단도 받아야 하는 조 대법원장 탄핵보다는 자진사퇴를 통해 같은 효과를 노리려는 전략으로 풀이된다.


## STEP 2 Query Transformation (질문 변환)

#### 벡터DB 불러오기 (01_NaiveRAG에서 저장한 것)

In [35]:
import chromadb
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

# 벡터DB 불러오기
chroma_client = chromadb.PersistentClient(path="./chroma_db")
collection = chroma_client.get_collection(name="news")
print(f"불러온 청크 수: {collection.count()}")

client = OpenAI()

불러온 청크 수: 476


#### Multi-Query 구현

In [36]:
def generate_multi_queries(question, n=3):
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": f"주어진 질문을 {n}가지 다른 표현으로 바꿔주세요. 각 줄에 하나씩만 출력하세요."},
             {"role": "user", "content": question}
        ]
    )
    queries = response.choices[0].message.content.strip().split("\n")
    return [q.strip().lstrip("0123456789.-) ") for q in queries if q.strip()]

# 테스트
question = "중동 상황 어때?"
queries = generate_multi_queries(question)
print(f"원본: {question}")
for i, q in enumerate(queries):
    print(f"  변환 {i+1}: {q}")

원본: 중동 상황 어때?
  변환 1: 현재 중동의 상황은 어떤가요?
  변환 2: 중동 지역의 현황은 어떤지 알려줄 수 있나요?
  변환 3: 지금 중동의 정세는 어떻게 돌아가고 있나요?


#### Multi-Query 검색

In [37]:
# Step 3에서 만든 collection이 있다고 가정
# 없으면 1단계 step3-vectordb 코드를 먼저 실행

def multi_query_search(question, collection, n_results=3):
    queries = generate_multi_queries(question)
    all_queries = [question] + queries  # 원본 포함

    all_docs = []
    seen_ids = set()

    for q in all_queries:
        q_resp = client.embeddings.create(input=[q], model="text-embedding-3-small")
        results = collection.query(query_embeddings=[q_resp.data[0].embedding], n_results=n_results)

        for doc_id, doc, meta in zip(results['ids'][0], results['documents'][0], results['metadatas'][0]):
            if doc_id not in seen_ids:
                seen_ids.add(doc_id)
                all_docs.append({"id": doc_id, "doc": doc, "meta": meta})

    return all_docs

results = multi_query_search("중동 상황 어때?", collection)
print(f"총 검색된 청크: {len(results)}개\n")
for r in results:
    print(f"[{r['meta']['category']}] {r['doc'][:80]}")
    print()

총 검색된 청크: 8개

[IT/과학] 중이다.

양 기관은 이번 협약을 통해 ▷넥스트라이즈-4YFN 간 공동 콘퍼런스 추진 ▷스타트업·투자자 교류 프로그램 신설 ▷오픈이노베이션 밋업

[사회] 씨가 회식을 마친 후 집에 바래다준다는 핑계로 집까지 따라와 성폭행했다"고 주장하며 세 차례에 걸쳐 언론 인터뷰를 통해 문제를 제기했습니다.



[사회] 정비 11억원 △동탄권 버스공영차고지 진입도로 재포장 공사 10억원 △황계지구 풍수해생활권 종합정비사업 14억원 등도 반영했다.

이번 추가경정

[사회] 담회에서 인사말을 통해 2026년 시정운영 방향 등을 설명하고 있다. (사진=연합뉴스)
오 시장은 5일 중구 서울시청 청사에서 ‘서남권 대개조 

[사회] 추경을 통해 시민 생활과 밀접한 현안 사업을 신속히 추진하고, 지역 균형 발전과 민생 안정 기반을 더욱 강화해 나가겠다"고 말했다.

[세계] 아가도록 추진할 것”이라고 밝혔다.

사우디와 UAE 외교장관은 분쟁 상황 속에서 중국의 역할을 요청했다. 알사우드 사우디 외교장관은 “분쟁이 

[정치] 하는 등 지역이 실질적 성장 거점이 되도록 집적화하겠다"며 "이전 대상 기관 전수조사와 지방정부 수요 조사 등을 통해 합리적이고 체계적인 로드맵

[정치]  경호·경비 업무 수행 지역이다. 그러나 국민주권정부(이재명 정부) 출범 이후 '열린 경호, 낮은 경호' 원칙에 따라 국민 이용에 불편함이 없도



#### HyDE 구현

In [38]:
def hyde_search(question, collection, n_results=3):
    # 1. 가상 답변 생성
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "질문에 대해 뉴스 기사 스타일로 가상의 답변을 1문단으로 작성하세요."},
            {"role": "user", "content": question}
        ]
    )
    hypothetical_doc = response.choices[0].message.content
    print(f"가상 답변: {hypothetical_doc[:100]}...\n")

    # 2. 가상 답변을 임베딩해서 검색
    h_resp = client.embeddings.create(input=[hypothetical_doc], model="text-embedding-3-small")
    results = collection.query(query_embeddings=[h_resp.data[0].embedding], n_results=n_results)

    return results

#### 일반 검색 vs Multi-Query vs HyDE

In [39]:
question = "경제에 미치는 영향은?"

print("=== 일반 검색 ===")
q_resp = client.embeddings.create(input=[question], model="text-embedding-3-small")
r1 = collection.query(query_embeddings=[q_resp.data[0].embedding], n_results=3)
for doc in r1['documents'][0]:
    print(f"  {doc[:80]}")

print("\n=== Multi-Query ===")
r2 = multi_query_search(question, collection)
for r in r2[:3]:
    print(f"  {r['doc'][:80]}")

print("\n=== HyDE ===")
r3 = hyde_search(question, collection)
for doc in r3['documents'][0]:
    print(f"  {doc[:80]}")

=== 일반 검색 ===
   기업 성장과 안정적인 경영 체계를 가능하게 하는 요소로 평가된다.

ESG 역시 규제 대응이 아닌 성장 전략으로 활용하고 있다. 탄소 배출 감
  털 전환(DX)을 통해 기업의 체질을 개선하고 장기적인 경쟁력을 확보하는 전략이다.

또 다른 특징은 소유와 경영의 명확한 분리다. 가문은 이사
  방식이다.

사회 환원 구조도 발렌베리 모델의 중요한 축이다. 기업에서 발생한 이익을 재단을 통해 기초 과학과 학술 연구, 사회 인프라에 재투자

=== Multi-Query ===
   기업 성장과 안정적인 경영 체계를 가능하게 하는 요소로 평가된다.

ESG 역시 규제 대응이 아닌 성장 전략으로 활용하고 있다. 탄소 배출 감
  털 전환(DX)을 통해 기업의 체질을 개선하고 장기적인 경쟁력을 확보하는 전략이다.

또 다른 특징은 소유와 경영의 명확한 분리다. 가문은 이사
  방식이다.

사회 환원 구조도 발렌베리 모델의 중요한 축이다. 기업에서 발생한 이익을 재단을 통해 기초 과학과 학술 연구, 사회 인프라에 재투자

=== HyDE ===
가상 답변: 최근의 경제 분석 보고서에 따르면, 세계 여러 국가에서 발생하는 공급망의 혼잡과 인플레이션 상승이 글로벌 경제에 심각한 영향을 미치고 있다. 전문가들은 이러한 요인이 소비자 물가 ...

   시장의 공급 차질 우려가 커지면서 싱가포르 상품거래소의 국제 석유제품 가격도 덩달아 상승하고 있습니다.

여기에 국내 유통업자들이 사태 이전 
   이날 다시 상승세로 돌아선 모습이다.

이 같은 반등 흐름은 간밤 미국 증시가 상승 마감한 영향으로 풀이된다. 미국 증시는 다우지수는 0.5%
  등으로 주식시장을 교란한 27곳 기업, 관련자 200여명에 대한 세무조사를 통해 총 2576억원을 세액을 추징했다고 5일 밝혔다. 코스피 상장기


## STEP 3 Re-ranking

In [40]:
import chromadb
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

chroma_client = chromadb.PersistentClient(path="./chroma_db")
collection = chroma_client.get_collection(name="news")
client = OpenAI()

In [41]:
question = "이란 전쟁의 경제적 영향은?"

q_resp = client.embeddings.create(input=[question], model="text-embedding-3-small")
results = collection.query(
    query_embeddings=[q_resp.data[0].embedding],
    n_results=10  # 넉넉하게 10개
)

print("=== 벡터 검색 Top 10 ===")
for i, (doc, meta) in enumerate(zip(results['documents'][0], results['metadatas'][0])):
    print(f"{i+1}. [{meta['category']}] {doc[:60]}")

=== 벡터 검색 Top 10 ===
1. [정치] HD현대는 정 회장의 이번 방문이 양국의 협력을 한층 끌어올리는 계기가 될 것으로 보고 있다.

5일 HD현
2. [사회] 기술을 바탕으로 글로벌 시장에서 경쟁력을 확보할 수 있도록 현장 중심의 지원을 이어가겠다"고 말했다.
3. [IT/과학] 털 전환(DX)을 통해 기업의 체질을 개선하고 장기적인 경쟁력을 확보하는 전략이다.

또 다른 특징은 소유와
4. [경제] 다.

전날 코스피는 미국과 이란 간 전쟁 발발 여파로 698.37포인트(12.06%) 급락, 역대 최대 낙
5. [경제]  회복했다.

이날 오전 9시 1분 코스피는 전장보다 157.38포인트(3.09%) 오른 5250.92로 출
6. [세계] 상에서 안전하다고 생각한 이란 전함을 미국 잠수함이 침몰시켰다"며 "제2차 세계대전 이후 어뢰로 적군 함정을
7. [사회]  △첨단산업 거점 조성 △사통팔달 교통체계 확립 △신속한 주택공급 △녹지축 연계 확산 등 4개 전략을 추진한
8. [경제]  시장의 공급 차질 우려가 커지면서 싱가포르 상품거래소의 국제 석유제품 가격도 덩달아 상승하고 있습니다.


9. [세계] 없이 추가적인 대이란 군사작전을 중단하도록 요구하는 내용의 결의안을 표결에 부칠 예정이다.

1973년 전쟁
10. [세계] 군, 해군, 리더십 모두 사라졌다. 그들은 대화를 원한다. 나는 '너무 늦었다'라고 말했다"라고 썼다.

N


In [42]:
def rerank(question, documents, top_k=3):
    # 문서 목록을 번호와 함께 구성
    doc_list = ""
    for i, doc in enumerate(documents):
        doc_list += f"[문서 {i + 1}] {doc}\n\n"

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": f"""질문에 가장 관련성 높은 문서 {top_k}개를 골라 번호만 출력하세요.
  관련성 높은 순서대로, 쉼표로 구분하세요.
  예시: 3,7,1"""},
            {"role": "user", "content": f"질문: {question}\n\n{doc_list}"}
        ]
    )

    # 번호 파싱
    answer = response.choices[0].message.content.strip()
    indices = [int(x.strip()) - 1 for x in answer.split(",") if x.strip().isdigit()]
    return indices

    # Re-ranking 실행


indices = rerank(question, results['documents'][0])

print(f"\n=== Re-ranking 후 Top 3 ===")
for rank, idx in enumerate(indices):
    doc = results['documents'][0][idx]
    meta = results['metadatas'][0][idx]
    print(f"{rank + 1}. (원래 {idx + 1}위) [{meta['category']}] {doc[:80]}")
    print()


=== Re-ranking 후 Top 3 ===
1. (원래 4위) [경제] 다.

전날 코스피는 미국과 이란 간 전쟁 발발 여파로 698.37포인트(12.06%) 급락, 역대 최대 낙폭과 하락률을 기록한 바 있다.

앞

2. (원래 8위) [경제]  시장의 공급 차질 우려가 커지면서 싱가포르 상품거래소의 국제 석유제품 가격도 덩달아 상승하고 있습니다.

여기에 국내 유통업자들이 사태 이전 

3. (원래 6위) [세계] 상에서 안전하다고 생각한 이란 전함을 미국 잠수함이 침몰시켰다"며 "제2차 세계대전 이후 어뢰로 적군 함정을 격침한 첫 사례"라고 말했습니다.




In [43]:
def rag_with_rerank(question, n_candidates=10, top_k=3):
    # 1. 벡터 검색으로 후보 확보
    q_resp = client.embeddings.create(input=[question], model="text-embedding-3-small")
    results = collection.query(query_embeddings=[q_resp.data[0].embedding], n_results=n_candidates)

    # 2. Re-ranking
    indices = rerank(question, results['documents'][0], top_k=top_k)
    reranked_docs = [results['documents'][0][i] for i in indices]

    # 3. LLM 답변 생성
    context = "\n\n".join([f"[참고자료 {i + 1}]\n{doc}" for i, doc in enumerate(reranked_docs)])
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "참고자료를 기반으로 답변하세요. 없는 내용은 '해당 정보를 찾을 수 없습니다'라고 하세요."},
             {"role": "user", "content": f"{context}\n\n질문: {question}"}
        ]
    )
    return response.choices[0].message.content


question = "이란 전쟁의 경제적 영향은?"

print("=== Re-ranking 없이 (Top 3) ===")
q_resp = client.embeddings.create(input=[question], model="text-embedding-3-small")
r = collection.query(query_embeddings=[q_resp.data[0].embedding], n_results=3)
context = "\n\n".join([f"[참고자료 {i + 1}]\n{doc}" for i, doc in enumerate(r['documents'][0])])
resp = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "참고자료를 기반으로 답변하세요."},
        {"role": "user", "content": f"{context}\n\n질문: {question}"}
    ]
)
print(resp.choices[0].message.content[:200])

print("\n=== Re-ranking 후 (Top 3) ===")
answer = rag_with_rerank(question)
print(answer[:200])

=== Re-ranking 없이 (Top 3) ===
이란 전쟁(이란-이라크 전쟁)은 1980년대 초부터 1988년까지 지속된 갈등으로, 양국 모두에 심각한 경제적 영향을 미쳤습니다. 전쟁의 경제적 영향은 다음과 같이 요약할 수 있습니다.

1. **직접적인 군사 비용**: 전쟁으로 인해 이란과 이라크 모두 막대한 군사 지출이 발생했습니다. 이는 국가 예산의 대부분을 차지하게 되었고, 다른 경제 부문에 대한 

=== Re-ranking 후 (Top 3) ===
이란 전쟁의 경제적 영향으로는 코스피 지수의 급락과 국제 석유제품 가격 상승이 있습니다. 코스피는 전날 미국과 이란 간 전쟁 발발 여파로 698.37포인트(12.06%) 급락하며 역대 최대 낙폭과 하락률을 기록했습니다. 이러한 급락은 전날 하루에만 1150.59포인트(19.3%)의 하락을 보였고, 이 후 다시 상승세로 돌아서는 모습이 나타났습니다.

또한,


## STEP 4 Hybrid Search (BM25 + 벡터 검색 결합)

In [44]:
# 터미널에서: uv add rank-bm25

import chromadb
from openai import OpenAI
from dotenv import load_dotenv
from rank_bm25 import BM25Okapi
import numpy as np

load_dotenv()

chroma_client = chromadb.PersistentClient(path="./chroma_db")
collection = chroma_client.get_collection(name="news")
client = OpenAI()

# 저장된 전체 문서 가져오기
all_data = collection.get(include=["documents", "metadatas"])
all_docs = all_data['documents']
all_metas = all_data['metadatas']
print(f"총 문서 수: {len(all_docs)}")

총 문서 수: 476


#### BM25 인덱스 생성

In [45]:
# 한국어는 띄어쓰기로 토큰화 (간단 버전)
tokenized_docs = [doc.split() for doc in all_docs]
bm25 = BM25Okapi(tokenized_docs)

# BM25 검색 테스트
question = "천궁 요격 성공률"
tokenized_query = question.split()
bm25_scores = bm25.get_scores(tokenized_query)

# 상위 5개
top_indices = np.argsort(bm25_scores)[::-1][:5]

print("=== BM25 검색 결과 ===")
for rank, idx in enumerate(top_indices):
    print(f"{rank+1}. (점수: {bm25_scores[idx]:.2f}) {all_docs[idx][:80]}")
    print()

=== BM25 검색 결과 ===
1. (점수: 5.64) 도로 날아드는 미사일과 공격용 드론이 요격용 미사일에 의해 산산조각이 납니다.

아랍에미리트가 공개한 요격 영상입니다.

개전 이후, 이란은 아

2. (점수: 0.00)  외교전에 뛰어든 것으로 해석된다.

5일 중국 외교부에 따르면 왕 부장은 이날 파이살 빈 파르한 알사우드 사우디아라비아 외교장관, 압둘라 빈 

3. (점수: 0.00) 왕이, 걸프국 잇따라 통화
“민간인 공격 강력히 규탄”
베이징=박세희 특파원

왕이(王毅) 중국 외교부장이 걸프 국가 외교 수장들과 잇따라 통화

4. (점수: 0.00) 강조했다.

5. (점수: 0.00)  우리를 공격했을 것”이라고 했다.

특히 “내가 4년 전에 오바마 (전 대통령이 맺은) 핵 협정을 종료하지 않았다면 이란은 이미 핵무기를 보유



#### 벡터 검색과 비교

In [46]:
  # 같은 질문으로 벡터 검색
q_resp = client.embeddings.create(input=[question], model="text-embedding-3-small")
vec_results = collection.query(query_embeddings=[q_resp.data[0].embedding], n_results=5)

print("=== 벡터 검색 결과 ===")
for i, doc in enumerate(vec_results['documents'][0]):
    print(f"{i+1}. {doc[:80]}")
    print()

=== 벡터 검색 결과 ===
1.  가정하에, 현실적인 주총 출석률 90%를 적용해 계산해보자. 이 경우 1석을 확보하기 위해서는 전체 의결권의 12.8%가 필요하다는 계산이 나

2. 방부는 설명했습니다.

이란 특유의 섞어쏘기 방식이 통하지 않은 건데, 아랍에미리트 방공망에서 우리 기술로 만든 '천궁-II'가 역할을 한 것으

3. 전 배치된 것으로 전해집니다.

중거리 지대공 유도무기인 '천궁-II'는 고도 약 15~20㎞의 미사일을 요격하기 위한 것으로, 360도 전방향

4. .

최 회장 등 사측은 지분율이 27.9%라는 점을 고려하면 2석을 안정적으로 확보할 수 있을 것으로 추산된다. 1석은 사실상 최 회장이 차지

5. 찬반 여부를 확정했다.

영풍-MBK는 크루서블조인트벤처(Crucible JV LLC) 측이 기타비상무이사 후보로 추천한 월터 필드 맥랠런에 대



#### 결과 합치기

In [47]:
def hybrid_search(question, collection, all_docs, all_metas, bm25, top_k=5, alpha=0.5):
    """
    alpha: 벡터 검색 가중치 (0~1)
    1-alpha: BM25 가중치
    alpha=0.5면 반반, alpha=0.7이면 벡터 검색 70% + BM25 30%
    """
    # 1. BM25 점수
    tokenized_query = question.split()
    bm25_scores = bm25.get_scores(tokenized_query)
    # 정규화 (0~1)
    if max(bm25_scores) > 0:
        bm25_scores = bm25_scores / max(bm25_scores)

    # 2. 벡터 검색 점수
    q_resp = client.embeddings.create(input=[question], model="text-embedding-3-small")
    vec_results = collection.query(
        query_embeddings=[q_resp.data[0].embedding],
        n_results=len(all_docs),
        include=["distances", "documents"]
    )

    # 거리 → 유사도로 변환, 정규화
    vec_scores = np.zeros(len(all_docs))
    for i, doc_id in enumerate(vec_results['ids'][0]):
        idx = all_data['ids'].index(doc_id)
        vec_scores[idx] = 1 - vec_results['distances'][0][i]  # 거리 → 유사도
    if max(vec_scores) > 0:
        vec_scores = vec_scores / max(vec_scores)

    # 3. 점수 합산
    combined_scores = alpha * vec_scores + (1 - alpha) * bm25_scores

    # 상위 k개
    top_indices = np.argsort(combined_scores)[::-1][:top_k]

    results = []
    for idx in top_indices:
        results.append({
            "doc": all_docs[idx],
            "meta": all_metas[idx],
            "score": combined_scores[idx],
            "vec_score": vec_scores[idx],
            "bm25_score": bm25_scores[idx]
        })
    return results


results = hybrid_search(question, collection, all_docs, all_metas, bm25)

print(f"=== Hybrid Search: '{question}' ===\n")
for i, r in enumerate(results):
    print(
        f"{i + 1}. [합산: {r['score']:.3f}] (벡터: {r['vec_score']:.3f}, BM25: {r['bm25_score']:.3f})")
    print(f"   [{r['meta']['category']}] {r['doc'][:70]}")
    print()

=== Hybrid Search: '천궁 요격 성공률' ===

1. [합산: 0.244] (벡터: -0.512, BM25: 1.000)
   [정치] 도로 날아드는 미사일과 공격용 드론이 요격용 미사일에 의해 산산조각이 납니다.

아랍에미리트가 공개한 요격 영상입니다.

개전

2. [합산: -0.103] (벡터: -0.206, BM25: 0.000)
   [경제]  가정하에, 현실적인 주총 출석률 90%를 적용해 계산해보자. 이 경우 1석을 확보하기 위해서는 전체 의결권의 12.8%가 필

3. [합산: -0.161] (벡터: -0.321, BM25: 0.000)
   [정치] 방부는 설명했습니다.

이란 특유의 섞어쏘기 방식이 통하지 않은 건데, 아랍에미리트 방공망에서 우리 기술로 만든 '천궁-II'

4. [합산: -0.179] (벡터: -0.357, BM25: 0.000)
   [정치] 전 배치된 것으로 전해집니다.

중거리 지대공 유도무기인 '천궁-II'는 고도 약 15~20㎞의 미사일을 요격하기 위한 것으로

5. [합산: -0.181] (벡터: -0.362, BM25: 0.000)
   [경제] .

최 회장 등 사측은 지분율이 27.9%라는 점을 고려하면 2석을 안정적으로 확보할 수 있을 것으로 추산된다. 1석은 사실



In [48]:
for alpha in [0.0, 0.3, 0.5, 0.7, 1.0]:
    results = hybrid_search(question, collection, all_docs, all_metas, bm25, alpha=alpha)
    print(f"--- alpha={alpha} (벡터 {int(alpha * 100)}% + BM25 {int((1 - alpha) * 100)}%) ---")
    for r in results[:3]:
        print(f"  {r['doc'][:60]}")
    print()

--- alpha=0.0 (벡터 0% + BM25 100%) ---
  도로 날아드는 미사일과 공격용 드론이 요격용 미사일에 의해 산산조각이 납니다.

아랍에미리트가 공개한 요격 
   외교전에 뛰어든 것으로 해석된다.

5일 중국 외교부에 따르면 왕 부장은 이날 파이살 빈 파르한 알사우드 
  왕이, 걸프국 잇따라 통화
“민간인 공격 강력히 규탄”
베이징=박세희 특파원

왕이(王毅) 중국 외교부장이 

--- alpha=0.3 (벡터 30% + BM25 70%) ---
  도로 날아드는 미사일과 공격용 드론이 요격용 미사일에 의해 산산조각이 납니다.

아랍에미리트가 공개한 요격 
   가정하에, 현실적인 주총 출석률 90%를 적용해 계산해보자. 이 경우 1석을 확보하기 위해서는 전체 의결권
  방부는 설명했습니다.

이란 특유의 섞어쏘기 방식이 통하지 않은 건데, 아랍에미리트 방공망에서 우리 기술로 

--- alpha=0.5 (벡터 50% + BM25 50%) ---
  도로 날아드는 미사일과 공격용 드론이 요격용 미사일에 의해 산산조각이 납니다.

아랍에미리트가 공개한 요격 
   가정하에, 현실적인 주총 출석률 90%를 적용해 계산해보자. 이 경우 1석을 확보하기 위해서는 전체 의결권
  방부는 설명했습니다.

이란 특유의 섞어쏘기 방식이 통하지 않은 건데, 아랍에미리트 방공망에서 우리 기술로 

--- alpha=0.7 (벡터 70% + BM25 30%) ---
  도로 날아드는 미사일과 공격용 드론이 요격용 미사일에 의해 산산조각이 납니다.

아랍에미리트가 공개한 요격 
   가정하에, 현실적인 주총 출석률 90%를 적용해 계산해보자. 이 경우 1석을 확보하기 위해서는 전체 의결권
  방부는 설명했습니다.

이란 특유의 섞어쏘기 방식이 통하지 않은 건데, 아랍에미리트 방공망에서 우리 기술로 

--- alpha=1.0 (벡터 100% + BM25 0%) ---
   가정하에, 현실적인 주총 출석률 90%를 적용해 계산해보자. 이 경우 1석을 확보하기

## STEP 5 Advanced RAG 통합 파이프라인

In [49]:
import chromadb
import numpy as np
from openai import OpenAI
from dotenv import load_dotenv
from rank_bm25 import BM25Okapi

load_dotenv()
client = OpenAI()

chroma_client = chromadb.PersistentClient(path="./chroma_db")
collection = chroma_client.get_collection(name="news")

all_data = collection.get(include=["documents", "metadatas"])
all_docs = all_data['documents']
all_metas = all_data['metadatas']

tokenized_docs = [doc.split() for doc in all_docs]
bm25 = BM25Okapi(tokenized_docs)

print(f"준비 완료! 청크 수: {len(all_docs)}")

준비 완료! 청크 수: 476


In [50]:
# 1. Multi-Query
def generate_multi_queries(question, n=3):
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": f"주어진 질문을 {n}가지 다른 표현으로 바꿔주세요. 각 줄에 하나씩만 출력하세요."},
            {"role": "user", "content": question}
        ]
    )
    queries = response.choices[0].message.content.strip().split("\n")
    return [q.strip().lstrip("0123456789.-) ") for q in queries if q.strip()]


# 2. Hybrid Search
def hybrid_search(question, top_k=10, alpha=0.5):
    tokenized_query = question.split()
    bm25_scores = bm25.get_scores(tokenized_query)
    if max(bm25_scores) > 0:
        bm25_scores = bm25_scores / max(bm25_scores)

    q_resp = client.embeddings.create(input=[question], model="text-embedding-3-small")
    vec_results = collection.query(
        query_embeddings=[q_resp.data[0].embedding],
        n_results=len(all_docs),
        include=["distances"]
    )

    vec_scores = np.zeros(len(all_docs))
    for i, doc_id in enumerate(vec_results['ids'][0]):
        idx = all_data['ids'].index(doc_id)
        vec_scores[idx] = 1 - vec_results['distances'][0][i]
    if max(vec_scores) > 0:
        vec_scores = vec_scores / max(vec_scores)

    combined = alpha * vec_scores + (1 - alpha) * bm25_scores
    top_indices = np.argsort(combined)[::-1][:top_k]

    return [{"doc": all_docs[i], "meta": all_metas[i], "score": combined[i]} for i in top_indices]


# 3. Re-ranking
def rerank(question, documents, top_k=3):
    doc_list = ""
    for i, doc in enumerate(documents):
        doc_list += f"[문서 {i + 1}] {doc}\n\n"

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system",
             "content": f"질문에 가장 관련성 높은 문서 {top_k}개를 골라 번호만 출력하세요. 관련성 높은 순서대로, 쉼표로 구분. 예: 3,7,1"},
            {"role": "user", "content": f"질문: {question}\n\n{doc_list}"}
        ]
    )
    answer = response.choices[0].message.content.strip()
    return [int(x.strip()) - 1 for x in answer.split(",") if x.strip().isdigit()]


In [51]:
def advanced_rag(question):
    print(f"질문: {question}\n")

    # Step 1: Multi-Query로 질문 확장
    queries = [question] + generate_multi_queries(question)
    print(f"[1] Multi-Query: {len(queries)}개 질문 생성")

    # Step 2: 각 질문으로 Hybrid Search → 결과 합치기
    all_results = []
    seen_docs = set()
    for q in queries:
        results = hybrid_search(q, top_k=5)
        for r in results:
            doc_key = r['doc'][:50]
            if doc_key not in seen_docs:
                seen_docs.add(doc_key)
                all_results.append(r)
    print(f"[2] Hybrid Search: {len(all_results)}개 후보 확보")

    # Step 3: Re-ranking으로 Top 3 선별
    candidate_docs = [r['doc'] for r in all_results[:15]]  # 최대 15개만
    top_indices = rerank(question, candidate_docs, top_k=3)
    final_docs = [candidate_docs[i] for i in top_indices]
    print(f"[3] Re-ranking: Top {len(final_docs)}개 선별\n")

    # Step 4: LLM 답변 생성
    context = "\n\n".join([f"[참고자료 {i + 1}]\n{doc}" for i, doc in enumerate(final_docs)])
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "참고자료를 기반으로 답변하세요. 없는 내용은 '해당 정보를 찾을 수 없습니다'라고 하세요."},
             {"role": "user", "content": f"{context}\n\n질문: {question}"}
        ]
    )

    return response.choices[0].message.content, final_docs

In [52]:
answer, docs = advanced_rag("이란 전쟁의 경제적 영향은?")
print("=== 답변 ===")
print(answer)

질문: 이란 전쟁의 경제적 영향은?

[1] Multi-Query: 4개 질문 생성
[2] Hybrid Search: 10개 후보 확보
[3] Re-ranking: Top 3개 선별

=== 답변 ===
해당 정보를 찾을 수 없습니다.


In [53]:
def naive_rag(question):
    q_resp = client.embeddings.create(input=[question], model="text-embedding-3-small")
    results = collection.query(query_embeddings=[q_resp.data[0].embedding], n_results=3)
    context = "\n\n".join([f"[참고자료 {i+1}]\n{doc}" for i, doc in enumerate(results['documents'][0])])
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "참고자료를 기반으로 답변하세요. 없는 내용은 '해당 정보를 찾을 수 없습니다'라고 하세요."},
             {"role": "user", "content": f"{context}\n\n질문: {question}"}
        ]
    )
    return response.choices[0].message.content

questions = [
    "이란 전쟁의 경제적 영향은?",
    "천궁 미사일 요격 성공률은?",
    "AI 관련 뉴스 알려줘",
]

for q in questions:
    print(f"Q: {q}")
    print(f"\n[Naive RAG]")
    print(naive_rag(q)[:150])
    print(f"\n[Advanced RAG]")
    answer, _ = advanced_rag(q)
    print(answer[:150])
    print("=" * 60)

Q: 이란 전쟁의 경제적 영향은?

[Naive RAG]
해당 정보를 찾을 수 없습니다.

[Advanced RAG]
질문: 이란 전쟁의 경제적 영향은?

[1] Multi-Query: 4개 질문 생성
[2] Hybrid Search: 11개 후보 확보
[3] Re-ranking: Top 3개 선별

해당 정보를 찾을 수 없습니다.
Q: 천궁 미사일 요격 성공률은?

[Naive RAG]
해당 정보를 찾을 수 없습니다.

[Advanced RAG]
질문: 천궁 미사일 요격 성공률은?

[1] Multi-Query: 4개 질문 생성
[2] Hybrid Search: 13개 후보 확보
[3] Re-ranking: Top 3개 선별

해당 정보를 찾을 수 없습니다.
Q: AI 관련 뉴스 알려줘

[Naive RAG]
해당 정보를 찾을 수 없습니다.

[Advanced RAG]
질문: AI 관련 뉴스 알려줘

[1] Multi-Query: 4개 질문 생성
[2] Hybrid Search: 20개 후보 확보
[3] Re-ranking: Top 3개 선별

마이크로소프트가 26일 서울 강남구 코엑스에서 '마이크로소프트 AI 투어'를 개최합니다. 이 행사에서는 기업, 파트너, 비즈니스 리더, 기술 실무자들이 AI 혁신을 이해하고 적용하는 데 필요한 인사이트를 제공할 예정입니다. 기조연설에는 스콧 거스리 마이크로소프트 클라우
